# Next-Word Prediction with LSTM (efficient version)
Full pipeline: load text -> clean -> tokenize -> build sequences -> train -> evaluate -> predict -> save.

**How to use:** put your training text in a file called `corpus.txt` next to this notebook
(the bigger and cleaner, the better). If the file is missing, a tiny sample text is used
just so the notebook runs end to end.

In [ ]:
import os
import re
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Settings
Everything you may want to tune is here.

In [ ]:
CORPUS_PATH   = "corpus.txt"
MAX_VOCAB     = 10000   # keep only the most frequent words
MAX_SEQ_LEN   = 20      # cap on (context + target) length; keeps padding small and training fast
EMBEDDING_DIM = 100
LSTM_UNITS    = 128
DROPOUT_RATE  = 0.3
BATCH_SIZE    = 128
EPOCHS        = 50
VAL_FRACTION  = 0.1     # share of sentences held out for validation

PAD_INDEX = 0           # Keras reserves 0 for padding
OOV_INDEX = 1           # index of the "<unk>" token (first token when oov_token is set)

## 2. Load and clean the text

In [ ]:
SAMPLE_TEXT = """
the quick brown fox jumps over the lazy dog.
the quick brown fox runs through the green forest.
the lazy dog sleeps under the old oak tree.
a quick brown rabbit hides in the tall grass.
the green forest is quiet in the early morning.
the old oak tree stands near the small river.
the small river flows through the green forest.
a lazy dog sleeps in the warm sun.
the quick fox jumps over the small river.
the brown rabbit runs through the tall grass.
the warm sun rises over the green forest.
the early morning is quiet and calm.
the dog runs after the quick brown rabbit.
the fox hides under the old oak tree.
a calm river flows near the quiet forest.
the tall grass moves in the warm wind.
"""

if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH, encoding="utf-8") as f:
        raw_text = f.read()
    print(f"Loaded {CORPUS_PATH}: {len(raw_text):,} characters")
else:
    raw_text = SAMPLE_TEXT
    print("corpus.txt not found -> using the tiny SAMPLE_TEXT (results will be weak).")


def clean_text(text):
    """Lowercase and keep only letters, digits, apostrophes and sentence-ending marks."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9'.!?\n\s]", " ", text)
    return re.sub(r"[ \t]+", " ", text)


sentences = [
    " ".join(s.split())
    for s in re.split(r"[.!?\n]+", clean_text(raw_text))
    if len(s.split()) >= 2
]
print(f"{len(sentences):,} sentences")
assert len(sentences) >= 10, "Need at least 10 sentences of text to train and validate."

## 3. Train / validation split (by sentence)
Splitting *before* building sequences prevents the same sentence from leaking into both sets.

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(sentences))
n_val = max(1, int(len(sentences) * VAL_FRACTION))

val_sentences   = [sentences[i] for i in order[:n_val]]
train_sentences = [sentences[i] for i in order[n_val:]]
print(f"train sentences: {len(train_sentences):,} | validation sentences: {len(val_sentences):,}")

## 4. Tokenize
The tokenizer is fitted on the training sentences only. Rare or unseen words become `<unk>`.

In [ ]:
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<unk>", filters="")
tokenizer.fit_on_texts(train_sentences)

word_index = tokenizer.word_index
vocabulary_size = min(MAX_VOCAB, len(word_index) + 1)
reverse_word_index = {i: w for w, i in word_index.items() if i < vocabulary_size}

longest = max(len(s.split()) for s in train_sentences)
sequence_length = min(longest, MAX_SEQ_LEN)   # context length is sequence_length - 1

print(f"vocabulary_size = {vocabulary_size:,}")
print(f"sequence_length = {sequence_length} (inputs have {sequence_length - 1} tokens)")

## 5. Build input/target pairs
For a sentence like `a b c d` we create: `a -> b`, `a b -> c`, `a b c -> d`.
Targets are stored as **integers** (not one-hot), which is the main memory/speed saving.

In [ ]:
def build_xy(sents):
    sequences = []
    for tokens in tokenizer.texts_to_sequences(sents):
        for i in range(1, len(tokens)):
            start = max(0, i + 1 - sequence_length)
            sequences.append(tokens[start:i + 1])
    padded = pad_sequences(sequences, maxlen=sequence_length, padding="pre")
    X, y = padded[:, :-1], padded[:, -1]
    keep = y != OOV_INDEX          # no point learning to predict "<unk>"
    return X[keep].astype("int32"), y[keep].astype("int32")


X_train, y_train = build_xy(train_sentences)
X_val,   y_val   = build_xy(val_sentences)
print(f"X_train {X_train.shape} | y_train {y_train.shape}")
print(f"X_val   {X_val.shape} | y_val   {y_val.shape}")


def make_ds(X, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(X_train, y_train, shuffle=True)
val_ds   = make_ds(X_val, y_val, shuffle=False)

## 6. Model
Changes vs. the original: padding is masked (`mask_zero=True`), dropout added, sparse labels.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocabulary_size, output_dim=EMBEDDING_DIM, mask_zero=True),
    Dropout(DROPOUT_RATE),
    LSTM(LSTM_UNITS),
    Dropout(DROPOUT_RATE),
    Dense(vocabulary_size, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")],
)

model.build(input_shape=(None, sequence_length - 1))
model.summary()

## 7. Train
Early stopping ends training when validation loss stops improving and restores the best weights.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].plot(history.history["val_top3_acc"], label="validation top-3", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_acc, val_top3 = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {val_loss:.3f} | perplexity: {np.exp(val_loss):.1f}")
print(f"Validation accuracy: {val_acc:.3f} | top-3 accuracy: {val_top3:.3f}")

## 9. Prediction
* `suggest_next_words` -> top-k suggestions with probabilities (like a phone keyboard)
* `generate_text` -> keeps predicting words; `temperature` controls randomness
  (low = safe/repetitive, high = creative/risky)

In [ ]:
def _encode(seed_text):
    tokens = tokenizer.texts_to_sequences([" ".join(clean_text(seed_text).split())])[0]
    if not tokens:
        raise ValueError("Seed text is empty after cleaning.")
    return pad_sequences([tokens], maxlen=sequence_length - 1, padding="pre")


def _next_word_probs(seed_text):
    probs = model.predict(_encode(seed_text), verbose=0)[0].astype("float64")
    probs[[PAD_INDEX, OOV_INDEX]] = 0.0        # never suggest padding or <unk>
    return probs / probs.sum()


def suggest_next_words(seed_text, top_k=3):
    probs = _next_word_probs(seed_text)
    top = np.argsort(probs)[-top_k:][::-1]
    return [(reverse_word_index.get(int(i), "<unk>"), float(probs[i])) for i in top]


def generate_text(seed_text, n_words=10, temperature=0.8):
    text = seed_text
    for _ in range(n_words):
        probs = _next_word_probs(text)
        logits = np.log(probs + 1e-12) / temperature
        p = np.exp(logits - logits.max())
        p[[PAD_INDEX, OOV_INDEX]] = 0.0
        p /= p.sum()
        text += " " + reverse_word_index.get(int(np.random.choice(len(p), p=p)), "<unk>")
    return text


seed = "the quick brown"
print(f"Seed phrase: '{seed}'")
for word, prob in suggest_next_words(seed, top_k=3):
    print(f"  {word:<12} {prob:.1%}")
print("Generated:", generate_text(seed, n_words=8))

## 10. Save and reload
Save the model plus the tokenizer and settings so you can predict later without retraining.

In [ ]:
model.save("next_word_lstm.keras")
with open("tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())
with open("config.json", "w") as f:
    json.dump({"sequence_length": sequence_length, "vocabulary_size": vocabulary_size}, f)
print("Saved: next_word_lstm.keras, tokenizer.json, config.json")

# To reload in a new session:
#   from tensorflow.keras.models import load_model
#   from tensorflow.keras.preprocessing.text import tokenizer_from_json
#   model = load_model("next_word_lstm.keras")
#   tokenizer = tokenizer_from_json(open("tokenizer.json", encoding="utf-8").read())
#   cfg = json.load(open("config.json")); sequence_length = cfg["sequence_length"]
#   reverse_word_index = {i: w for w, i in tokenizer.word_index.items() if i < cfg["vocabulary_size"]}